## Bước 1: Load dữ liệu và xem tổng quan

In [1]:
import pandas as pd

FILE_NAME = r'C:\Users\Admin\Documents\NEW_JOURNEY\PROJECT_3\online_retail_II.csv'  # sua duong dan neu can

df = pd.read_csv(FILE_NAME, encoding='ISO-8859-1')

print(f"So dong: {len(df)}")
print(f"So cot: {len(df.columns)}")
print(f"\nTen cac cot:\n{list(df.columns)}")
print(f"\nKieu du lieu tung cot:\n{df.dtypes}")

So dong: 1067371
So cot: 8

Ten cac cot:
['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']

Kieu du lieu tung cot:
Invoice         object
StockCode       object
Description     object
Quantity         int64
InvoiceDate     object
Price          float64
Customer ID    float64
Country         object
dtype: object


#Invoice: mã đơn hàng

#StockCode/Description: mã và tên sản phẩm

#Quantity, Price: số lượng và đơn giá của sản phẩm đó trong đơn hàng

#Customer ID: mã khách hàng (ai đã mua)

#InvoiceDate: thời điểm mua

## Bước 2: Kiểm tra missing data và dòng trùng lặp

In [2]:
print("So luong gia tri thieu tung cot:")
print(df.isna().sum())

print(f"\nSo dong trung lap hoan toan: {df.duplicated().sum()}")

So luong gia tri thieu tung cot:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

So dong trung lap hoan toan: 34335


## Bước 3: Kiểm tra kiểu dữ liệu thật của cột Invoice

In [3]:
print(f"Kieu du lieu cot Invoice: {df['Invoice'].dtype}")
print(f"\n5 gia tri Invoice mau:")
print(df['Invoice'].head())

Kieu du lieu cot Invoice: object

5 gia tri Invoice mau:
0    489434
1    489434
2    489434
3    489434
4    489434
Name: Invoice, dtype: object


## Bước 4: Điều tra các dòng có Quantity âm

In [4]:
df_negative_qty = df[df['Quantity'] < 0]

print(f"So dong co Quantity am: {len(df_negative_qty)}")
print(f"Ty le tren tong: {len(df_negative_qty) / len(df) * 100:.2f}%")

print("\n10 dong mau:")
print(df_negative_qty[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price']].head(10))

So dong co Quantity am: 22950
Ty le tren tong: 2.15%

10 dong mau:
     Invoice StockCode                        Description  Quantity  Price
178  C489449     22087           PAPER BUNTING WHITE LACE       -12   2.95
179  C489449    85206A       CREAM FELT EASTER EGG BASKET        -6   1.65
180  C489449     21895      POTTING SHED SOW 'N' GROW SET        -4   4.25
181  C489449     21896                 POTTING SHED TWINE        -6   2.10
182  C489449     22083         PAPER CHAIN KIT RETRO SPOT       -12   2.95
183  C489449     21871                SAVE THE PLANET MUG       -12   1.25
184  C489449     84946    ANTIQUE SILVER TEA GLASS ETCHED       -12   1.25
185  C489449    84970S  HANGING HEART ZINC T-LIGHT HOLDER       -24   0.85
186  C489449     22090          PAPER BUNTING RETRO SPOTS       -12   2.95
196  C489459    90200A         PURPLE SWEETHEART BRACELET        -3   4.25


## Bước 5: Điều tra các dòng có Price âm hoặc bằng 0

In [5]:
df_negative_price = df[df['Price'] < 0]
df_zero_price = df[df['Price'] == 0]

print(f"So dong co Price am: {len(df_negative_price)}")
print(df_negative_price[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price']])

print(f"\nSo dong co Price bang 0: {len(df_zero_price)}")
print(df_zero_price[['Invoice', 'StockCode', 'Description', 'Quantity', 'Price']].head(10))

So dong co Price am: 5
        Invoice StockCode      Description  Quantity     Price
179403  A506401         B  Adjust bad debt         1 -53594.36
276274  A516228         B  Adjust bad debt         1 -44031.79
403472  A528059         B  Adjust bad debt         1 -38925.87
825444  A563186         B  Adjust bad debt         1 -11062.06
825445  A563187         B  Adjust bad debt         1 -11062.06

So dong co Price bang 0: 6202
     Invoice StockCode   Description  Quantity  Price
263   489464     21733  85123a mixed       -96    0.0
283   489463     71477         short      -240    0.0
284   489467    85123A   21733 mixed      -192    0.0
470   489521     21646           NaN       -50    0.0
3114  489655     20683           NaN       -44    0.0
3161  489659     21350           NaN       230    0.0
3162  489660     35956          lost     -1043    0.0
3168  489663    35605A       damages      -117    0.0
3731  489781     84292           NaN        17    0.0
4296  489806     18010      

## Bước 6: Tìm quy luật ký tự đầu của Invoice (để phát hiện đơn hủy, điều chỉnh...)

In [6]:
# Lay ky tu dau tien cua Invoice (neu la chu) hoac danh dau 'so' neu toan bo la so
df['invoice_prefix'] = df['Invoice'].astype(str).str[0]
df['invoice_prefix'] = df['invoice_prefix'].apply(lambda x: x if not x.isdigit() else 'SO_(toan_la_so)')

print("Thong ke theo ky tu dau cua Invoice:")
print(df['invoice_prefix'].value_counts())

Thong ke theo ky tu dau cua Invoice:
invoice_prefix
SO_(toan_la_so)    1047871
C                    19494
A                        6
Name: count, dtype: int64


## Bước 7: Đối chiếu - các dòng Quantity âm có rơi vào nhóm prefix đặc biệt nào không?

In [7]:
print("Phan bo prefix Invoice trong CAC DONG CO QUANTITY AM:")
print(df[df['Quantity'] < 0]['invoice_prefix'].value_counts())

Phan bo prefix Invoice trong CAC DONG CO QUANTITY AM:
invoice_prefix
C                  19493
SO_(toan_la_so)     3457
Name: count, dtype: int64
